# Text-to-SQL em português: fine-tuning de LLM com QLoRA (GPU)

Notebook para rodar no **Google Colab** (Ambiente de execução → Alterar tipo → **T4 GPU**).
Fluxo: verificar GPU → dataset → baseline → 4 treinos com parâmetros diferentes → avaliação → frontend.

## 1. Verificar a GPU (print exigido no CP)

In [11]:
%cd /content/text2sql-llm
!git pull

/content/text2sql-llm
remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 8 (delta 5), reused 8 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 2.48 KiB | 508.00 KiB/s, done.
From https://github.com/palomari/text2sql-llm
   f0343ae..3893a54  main       -> origin/main
Updating f0343ae..3893a54
Fast-forward
 notebooks/text2sql_colab.ipynb | 129 +++++++++++++++++++++++------------------
 src/common.py                  |  16 +++++
 src/evaluate_model.py          |   9 ++-
 src/train.py                   |  19 +++---
 4 files changed, 106 insertions(+), 67 deletions(-)


In [1]:
!nvidia-smi
import torch
print("CUDA disponível:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__, "| CUDA:", torch.version.cuda)

Mon Oct 12 08:40:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Obter o código
Troque `SEU_USUARIO` pelo seu usuário do GitHub (o repositório precisa estar publicado).

In [3]:
GITHUB_REPO = "https://github.com/palomari/text2sql-llm.git"
!git clone {GITHUB_REPO}
%cd text2sql-llm

Cloning into 'text2sql-llm'...
remote: Enumerating objects: 22, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 22 (delta 3), reused 22 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (22/22), 88.46 KiB | 736.00 KiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/text2sql-llm


In [4]:
!pip install -q -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 21.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 21.0 MB/s eta 0:00:00:00:0100:01


In [12]:
!python src/train.py --name teste --epochs 0.2

GPU: Tesla T4
Precisão: fp16 | compute capability: (7, 5)
Loading weights: 100% 338/338 [00:03<00:00, 94.12it/s] 
Adding EOS to train dataset: 100% 1142/1142 [00:00<00:00, 31096.35 examples/s]
Tokenizing train dataset: 100% 1142/1142 [00:00<00:00, 1630.09 examples/s]
Building labels for train dataset: 100% 1142/1142 [00:00<00:00, 4272.78 examples/s]
Truncating train dataset: 100% 1142/1142 [00:00<00:00, 3883.49 examples/s]
Dropping fully masked examples from train dataset: 100% 1142/1142 [00:00<00:00, 5606.60 examples/s]
Adding EOS to eval dataset: 100% 125/125 [00:00<00:00, 24346.99 examples/s]
Tokenizing eval dataset: 100% 125/125 [00:00<00:00, 1544.03 examples/s]
Building labels for eval dataset: 100% 125/125 [00:00<00:00, 4298.25 examples/s]
Truncating eval dataset: 100% 125/125 [00:00<00:00, 3736.99 examples/s]
Dropping fully masked examples from eval dataset: 100% 125/125 [00:00<00:00, 5410.16 examples/s]
Parâmetros treináveis convertidos para fp32: 392
Dtypes treináveis: {'torch

## 3. Criar banco e dataset

In [ ]:
!python src/make_dataset.py
!head -n 3 data/train.jsonl

## 4. Baseline: modelo SEM fine-tuning

In [ ]:
!python src/evaluate_model.py --name baseline

## 5. Experimentos com parâmetros diferentes
Cada execução muda **um** parâmetro em relação à `run1`. Os resultados vão para `results/experimentos.csv`.

In [ ]:
!python src/train.py --name run1 --r 8  --lr 2e-4 --epochs 3

In [ ]:
!python src/train.py --name run2 --r 16 --lr 2e-4 --epochs 3

In [ ]:
!python src/train.py --name run3 --r 8  --lr 1e-4 --epochs 3

In [ ]:
!python src/train.py --name run4 --r 8  --lr 2e-4 --epochs 5

## 6. Avaliar cada execução no conjunto de teste

In [ ]:
for r in ["run1", "run2", "run3", "run4"]:
    !python src/evaluate_model.py --name {r} --adapter outputs/{r}/final

## 7. Parâmetros de inferência (temperature) na melhor execução
Troque `run1` pela melhor execução.

In [ ]:
BEST = "run1"
for t in [0.0, 0.3, 0.7, 1.0]:
    !python src/evaluate_model.py --name {BEST} --adapter outputs/{BEST}/final --temperature {t}

## 8. Tabelas e gráfico

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
exp = pd.read_csv("results/experimentos.csv")
av = pd.read_csv("results/avaliacao.csv")
display(exp)
display(av)

d = av[av.temperature == 0.0]
plt.figure(figsize=(7, 4))
plt.bar(d.run, d.exec_accuracy)
plt.ylabel("Accuracy por execução"); plt.title("Baseline vs. fine-tuning (teste)")
plt.ylim(0, 1); plt.tight_layout()
plt.savefig("results/comparacao.png", dpi=150)
plt.show()

## 9. Exemplos qualitativos (baseline x fine-tuned)

In [ ]:
import json
base = json.load(open("results/pred_baseline_T0.0.json", encoding="utf-8"))
ft = json.load(open(f"results/pred_{BEST}_T0.0.json", encoding="utf-8"))
for b, f in list(zip(base, ft))[:8]:
    print("Pergunta :", b["question"])
    print("Esperada :", b["gold_sql"])
    print("Baseline :", b["pred_sql"], "->", "OK" if b["correto"] else "ERRO")
    print("Fine-tune:", f["pred_sql"], "->", "OK" if f["correto"] else "ERRO")
    print("-" * 80)

## 10. Frontend (Gradio)
Gera um link público temporário. Tire o print do frontend funcionando.

In [ ]:
!python src/app.py --adapter outputs/{BEST}/final --share

## 11. Salvar resultados no Google Drive (a sessão do Colab é temporária)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/text2sql-llm
!cp -r results outputs /content/drive/MyDrive/text2sql-llm/